# 04i — Supervised Contrastive Learning: Tumor + Healthy

## Obiettivo

Migliorare la separazione multiclass dello spazio embedding delle FCGR
utilizzando Supervised Contrastive Learning.

Gli esperimenti precedenti hanno mostrato che:

- Euclidean Contrastive Loss produce una discreta separazione pairwise;
- il passaggio da 18 a 12 classi migliora la classificazione;
- un linear probe sugli embedding migliora solo marginalmente;
- riaddestrare Euclidean V3 direttamente sulle 12 classi non migliora
  la classificazione multiclass;
- Batch-Hard Triplet Loss produce representation collapse.

Questo suggerisce che il principale limite sia la geometria multiclass
appresa dall'encoder.

Supervised Contrastive Learning utilizza contemporaneamente tutti i
sample della stessa classe come positivi e quelli delle altre classi
come negativi.

## Task

12 classi:

- 11 tipi di tumore;
- Healthy.

## Configurazione iniziale

- FCGR k=6
- CNN V3 bias-free
- embedding 128D
- projection head per SupCon
- temperature = 0.07
- batch class-balanced
- 12 classi × 8 sample = 96 FCGR
- dynamic balanced sampling
- training da zero
- checkpoint tramite validation Macro-F1 multiclass
- test set non utilizzato durante il tuning

L'embedding utilizzato per la classificazione finale è quello prodotto
dall'encoder prima del projection head.

In [1]:
# ============================================================
# CELL 2 — IMPORT
# ============================================================

from pathlib import Path

import json
import random
import time
import copy
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)


print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch: 2.12.0+cu126
CUDA: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# ============================================================
# CELL 3 — PATH + CONFIG
# ============================================================

CURRENT_DIR = Path.cwd().resolve()


if CURRENT_DIR.name == "notebooks":

    PROJECT_ROOT = CURRENT_DIR.parent

else:

    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "supcon_v3_tumor_healthy"
)


ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)


CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)


VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)


PAIR_CONFIG_PATH = (
    PROCESSED_DIR
    / "siamese_pair_config.json"
)


with open(
    PAIR_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    pair_config = json.load(f)


K = int(
    pair_config["k"]
)


RANDOM_STATE = int(
    pair_config["random_state"]
)


FCGR_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}.npy"
)


FCGR_INDEX_PATH = (
    PROCESSED_DIR
    / "fcgr_cache"
    / f"fcgr_k{K}_index.tsv"
)


# ============================================================
# SUPCON CONFIG
# ============================================================

N_CLASSES = 12

SAMPLES_PER_CLASS = 8

BATCH_SIZE = (
    N_CLASSES
    *
    SAMPLES_PER_CLASS
)


EMBEDDING_DIM = 128

PROJECTION_DIM = 128

TEMPERATURE = 0.07


LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


print(
    "k:",
    K
)

print(
    "N classes:",
    N_CLASSES
)

print(
    "Samples/class:",
    SAMPLES_PER_CLASS
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Temperature:",
    TEMPERATURE
)

k: 6
N classes: 12
Samples/class: 8
Batch size: 96
Temperature: 0.07


In [3]:
# ============================================================
# CELL 4 — SEED + DEVICE
# ============================================================

def set_seed(
    seed
):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


set_seed(
    RANDOM_STATE
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )


AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print(
    "Device:",
    DEVICE
)

print(
    "AMP:",
    AMP_ENABLED
)

Device: cuda
AMP: True


In [4]:
# ============================================================
# CELL 5 — LOAD TUMOR + HEALTHY DATA
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)


metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# VALIDATION
# ============================================================

class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original[
            "class_id"
        ].isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = (
    val_metadata[
        "class_id"
    ]
)


val_metadata[
    "class_id"
] = (

    val_metadata[
        "original_class_id"
    ]
    .map(
        old_to_new
    )
    .astype(int)
)


print(
    "Train:",
    len(train_metadata)
)

print(
    "Validation:",
    len(val_metadata)
)

print(
    "Train classes:",
    train_metadata[
        "class_id"
    ].nunique()
)


display(
    train_metadata[
        "class_id"
    ]
    .value_counts()
    .sort_index()
    .rename("n_train")
    .to_frame()
)

Train: 96167
Validation: 9753
Train classes: 12


,n_train
class_id,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,5052


In [5]:
# ============================================================
# CELL 6 — FCGR
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print(
    "FCGR:",
    fcgr_memmap.shape,
    fcgr_memmap.dtype
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing val:",
    missing_val
)


assert missing_train == 0
assert missing_val == 0

FCGR: (150272, 64, 64) float32
Missing train: 0
Missing val: 0


In [6]:
# ============================================================
# CELL 7 — SINGLE SAMPLE DATASET
# ============================================================

class SingleFCGRDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )


        self.fcgr_memmap = (
            fcgr_memmap
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


    def __len__(
        self
    ):

        return len(
            self.metadata
        )


    def __getitem__(
        self,
        index
    ):

        row = int(
            self.rows[index]
        )


        fcgr = np.array(
            self.fcgr_memmap[row],
            dtype=np.float32,
            copy=True
        )


        return {

            "x":
                torch.from_numpy(
                    fcgr
                ).unsqueeze(0),

            "class_id":
                torch.tensor(
                    self.labels[index],
                    dtype=torch.long
                )
        }


train_dataset = SingleFCGRDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


print(
    "Train dataset:",
    len(train_dataset)
)

Train dataset: 96167


In [7]:
# ============================================================
# CELL 8 — DYNAMIC BALANCED PK SAMPLER
# ============================================================

class DynamicPKBatchSampler(Sampler):

    def __init__(
        self,
        labels,
        samples_per_class,
        seed=42
    ):

        self.labels = np.asarray(
            labels,
            dtype=np.int64
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.K = int(
            samples_per_class
        )


        self.seed = int(
            seed
        )


        self.class_to_indices = {

            int(class_id):

                np.where(
                    self.labels
                    ==
                    class_id
                )[0]

            for class_id
            in self.classes
        }


        self.min_class_size = min(

            len(indices)

            for indices
            in self.class_to_indices.values()
        )


        self.batches_per_epoch = (
            self.min_class_size
            //
            self.K
        )


        self.samples_per_class_per_epoch = (
            self.batches_per_epoch
            *
            self.K
        )


        self.epoch = 0


    def set_epoch(
        self,
        epoch
    ):

        self.epoch = int(
            epoch
        )


    def __len__(
        self
    ):

        return (
            self.batches_per_epoch
        )


    def __iter__(
        self
    ):

        rng = np.random.default_rng(

            self.seed
            +
            self.epoch
            *
            100_003
        )


        # ====================================================
        # NUOVO SUBSET DI OGNI CLASSE A OGNI EPOCA
        # ====================================================

        class_pools = {}


        for class_id in self.classes:

            class_id = int(
                class_id
            )


            shuffled = rng.permutation(

                self.class_to_indices[
                    class_id
                ]
            )


            class_pools[
                class_id
            ] = shuffled[
                :
                self.samples_per_class_per_epoch
            ]


        # ====================================================
        # BATCH
        # ====================================================

        for batch_idx in range(
            self.batches_per_epoch
        ):

            batch_indices = []


            start = (
                batch_idx
                *
                self.K
            )

            end = (
                start
                +
                self.K
            )


            for class_id in self.classes:

                class_id = int(
                    class_id
                )


                selected = (
                    class_pools[
                        class_id
                    ][
                        start:end
                    ]
                )


                batch_indices.extend(
                    selected.tolist()
                )


            rng.shuffle(
                batch_indices
            )


            yield (
                batch_indices
            )

In [8]:
# ============================================================
# CELL 9 — TRAIN LOADER SANITY CHECK
# ============================================================

train_sampler = DynamicPKBatchSampler(

    labels=
        train_dataset.labels,

    samples_per_class=
        SAMPLES_PER_CLASS,

    seed=
        RANDOM_STATE
)


train_loader = DataLoader(

    train_dataset,

    batch_sampler=
        train_sampler,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print("=" * 70)
print("DYNAMIC BALANCED SAMPLER")
print("=" * 70)

print(
    "Classi:",
    len(train_sampler.classes)
)

print(
    "K:",
    train_sampler.K
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Classe minima:",
    train_sampler.min_class_size
)

print(
    "Batch/epoch:",
    len(train_sampler)
)

print(
    "Sample/class/epoch:",
    train_sampler.samples_per_class_per_epoch
)

print(
    "Sample totali/epoch:",
    len(train_sampler)
    *
    BATCH_SIZE
)


batch_check = next(
    iter(train_loader)
)


x_check = (
    batch_check["x"]
)

y_check = (
    batch_check["class_id"]
)


classes_check, counts_check = torch.unique(

    y_check,

    return_counts=True
)


print()
print(
    "x:",
    x_check.shape
)

print(
    "Numero classi:",
    len(classes_check)
)

print(
    "Conteggi:",
    counts_check.tolist()
)


assert (
    len(classes_check)
    ==
    N_CLASSES
)


assert torch.all(
    counts_check
    ==
    SAMPLES_PER_CLASS
)


print()
print(
    "Dynamic balanced batch: OK"
)

DYNAMIC BALANCED SAMPLER
Classi: 12
K: 8
Batch size: 96
Classe minima: 2111
Batch/epoch: 263
Sample/class/epoch: 2104
Sample totali/epoch: 25248

x: torch.Size([96, 1, 64, 64])
Numero classi: 12
Conteggi: [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]

Dynamic balanced batch: OK


In [9]:
# ============================================================
# CELL 10 — SUPCON V3 MODEL
# ============================================================

class SupConV3(nn.Module):

    def __init__(
        self,
        embedding_dim=128,
        projection_dim=128
    ):

        super().__init__()


        # ====================================================
        # SAME V3 FEATURE EXTRACTOR
        # ====================================================

        self.features = nn.Sequential(

            nn.Conv2d(
                1, 32, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 32),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32, 32, 3,
                padding=1,
                bias=False
            ),

            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 128, 3,
                padding=1,
                bias=False
            ),

            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(
                (4, 4)
            )
        )


        # ====================================================
        # EMBEDDING
        # ====================================================

        self.embedding_head = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                256,
                embedding_dim
            )
        )


        # ====================================================
        # SUPCON PROJECTION HEAD
        # ====================================================

        self.projection_head = nn.Sequential(

            nn.Linear(
                embedding_dim,
                embedding_dim
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                embedding_dim,
                projection_dim
            )
        )


    def forward(
        self,
        x
    ):

        features = self.features(
            x
        )


        h_raw = self.embedding_head(
            features
        )


        # Embedding finale usato a valle
        h = F.normalize(
            h_raw,
            p=2,
            dim=1,
            eps=1e-8
        )


        # Projection usata soltanto dalla SupCon Loss
        z_raw = self.projection_head(
            h_raw
        )


        z = F.normalize(
            z_raw,
            p=2,
            dim=1,
            eps=1e-8
        )


        return (
            h,
            z
        )


model = SupConV3(

    embedding_dim=
        EMBEDDING_DIM,

    projection_dim=
        PROJECTION_DIM

).to(
    DEVICE
)


print(
    model
)

print()

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
)

SupConV3(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): GroupNorm(8, 32, eps=1e-05, affine=True, bias=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (7): GroupNorm(8, 64, eps=1e-05, affine=True, bias=True)
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): GroupNorm(8, 128, eps=1e-05, affine=True, bias=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (14): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), pa

In [10]:
# ============================================================
# CELL 11 — SUPERVISED CONTRASTIVE LOSS
# ============================================================

class SupervisedContrastiveLoss(nn.Module):

    def __init__(
        self,
        temperature=0.07
    ):

        super().__init__()

        self.temperature = float(
            temperature
        )


    def forward(
        self,
        projections,
        labels
    ):

        projections = (
            projections.float()
        )

        labels = (
            labels.long()
        )


        batch_size = (
            projections.shape[0]
        )


        # ====================================================
        # COSINE SIMILARITY
        #
        # z è già L2-normalizzato
        # ====================================================

        logits = torch.matmul(
            projections,
            projections.T
        )


        logits = (
            logits
            /
            self.temperature
        )


        # Stabilità numerica
        logits_max = (
            logits.max(
                dim=1,
                keepdim=True
            ).values
        )


        logits = (
            logits
            -
            logits_max.detach()
        )


        # ====================================================
        # MASK
        # ====================================================

        labels_matrix = (
            labels.unsqueeze(0)
            ==
            labels.unsqueeze(1)
        )


        self_mask = torch.eye(
            batch_size,
            dtype=torch.bool,
            device=labels.device
        )


        positive_mask = (
            labels_matrix
            &
            ~self_mask
        )


        logits_mask = (
            ~self_mask
        )


        # ====================================================
        # LOG PROBABILITY
        # ====================================================

        exp_logits = (

            torch.exp(
                logits
            )

            *
            logits_mask.float()
        )


        log_prob = (

            logits

            -
            torch.log(

                exp_logits
                .sum(
                    dim=1,
                    keepdim=True
                )

                +
                1e-12
            )
        )


        positives_per_anchor = (
            positive_mask
            .sum(
                dim=1
            )
        )


        assert torch.all(
            positives_per_anchor > 0
        )


        mean_log_prob_positive = (

            (
                positive_mask.float()
                *
                log_prob
            )
            .sum(
                dim=1
            )

            /
            positives_per_anchor
        )


        loss = (
            -mean_log_prob_positive
            .mean()
        )


        # ====================================================
        # DIAGNOSTICHE
        # ====================================================

        cosine_matrix = torch.matmul(
            projections,
            projections.T
        )


        negative_mask = (
            (~labels_matrix)
            &
            (~self_mask)
        )


        positive_cosine = (
            cosine_matrix[
                positive_mask
            ]
            .mean()
        )


        negative_cosine = (
            cosine_matrix[
                negative_mask
            ]
            .mean()
        )


        cosine_gap = (
            positive_cosine
            -
            negative_cosine
        )


        stats = {

            "positive_cosine":
                positive_cosine.detach(),

            "negative_cosine":
                negative_cosine.detach(),

            "cosine_gap":
                cosine_gap.detach()
        }


        return (
            loss,
            stats
        )


supcon_criterion = (
    SupervisedContrastiveLoss(
        temperature=
            TEMPERATURE
    )
)


print(
    "SupCon temperature:",
    supcon_criterion.temperature
)

SupCon temperature: 0.07


In [11]:
# ============================================================
# CELL 12 — SUPCON SANITY CHECK
# ============================================================

model.eval()


batch_check = next(
    iter(train_loader)
)


x = (
    batch_check["x"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


labels = (
    batch_check["class_id"]
    .to(
        DEVICE,
        non_blocking=True
    )
)


with torch.no_grad():

    with torch.autocast(

        device_type=
            DEVICE.type,

        dtype=
            torch.float16
            if DEVICE.type == "cuda"
            else torch.bfloat16,

        enabled=
            AMP_ENABLED

    ):

        h, z = model(
            x
        )


    (
        sanity_loss,
        sanity_stats
    ) = supcon_criterion(
        z,
        labels
    )


print("=" * 70)
print("SUPCON — INITIAL SPACE")
print("=" * 70)

print(
    "Loss:",
    f"{sanity_loss.item():.4f}"
)

print(
    "Positive cosine:",
    f"{sanity_stats['positive_cosine'].item():.4f}"
)

print(
    "Negative cosine:",
    f"{sanity_stats['negative_cosine'].item():.4f}"
)

print(
    "Cosine gap:",
    f"{sanity_stats['cosine_gap'].item():.4f}"
)

print(
    "Embedding norm:",
    f"{h.norm(dim=1).mean().item():.4f}"
)

print(
    "Projection norm:",
    f"{z.norm(dim=1).mean().item():.4f}"
)


assert torch.isfinite(
    sanity_loss
)

assert torch.allclose(

    h.norm(dim=1),

    torch.ones(
        h.shape[0],
        device=DEVICE
    ),

    atol=1e-4
)


assert torch.allclose(

    z.norm(dim=1),

    torch.ones(
        z.shape[0],
        device=DEVICE
    ),

    atol=1e-4
)


print()
print(
    "SupCon sanity check: OK"
)

SUPCON — INITIAL SPACE
Loss: 4.5416
Positive cosine: 0.9837
Negative cosine: 0.9822
Cosine gap: 0.0015
Embedding norm: 1.0000
Projection norm: 1.0000

SupCon sanity check: OK


In [12]:
# ============================================================
# CELL 13 — FIXED VALIDATION REFERENCE SET
# ============================================================

REFERENCE_PER_CLASS = 500

REFERENCE_SEED = (
    RANDOM_STATE
    +
    80_000
)


reference_parts = []


for class_id, group in train_metadata.groupby(
    "class_id"
):

    selected = group.sample(

        n=REFERENCE_PER_CLASS,

        replace=False,

        random_state=
            REFERENCE_SEED
            +
            int(class_id)
    )


    reference_parts.append(
        selected
    )


reference_metadata = (

    pd.concat(
        reference_parts,
        ignore_index=True
    )

    .reset_index(drop=True)
)


reference_dataset = SingleFCGRDataset(

    metadata=
        reference_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


val_dataset = SingleFCGRDataset(

    metadata=
        val_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


EVAL_BATCH_SIZE = 256


reference_loader = DataLoader(

    reference_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Reference:",
    len(reference_dataset)
)

print(
    "Reference/class:",
    REFERENCE_PER_CLASS
)

print(
    "Validation:",
    len(val_dataset)
)

Reference: 6000
Reference/class: 500
Validation: 9753


In [13]:
# ============================================================
# CELL 14 — EXTRACT ENCODER EMBEDDINGS
# ============================================================

def extract_h_embeddings(
    model,
    loader
):

    model.eval()


    embeddings_list = []
    labels_list = []


    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )


            with torch.autocast(

                device_type=
                    DEVICE.type,

                dtype=
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16,

                enabled=
                    AMP_ENABLED

            ):

                h, _ = model(
                    x
                )


            embeddings_list.append(

                h.float()
                .cpu()
                .numpy()
            )


            labels_list.append(

                batch[
                    "class_id"
                ]
                .numpy()
            )


    return (

        np.concatenate(
            embeddings_list,
            axis=0
        ),

        np.concatenate(
            labels_list,
            axis=0
        ).astype(
            np.int64
        )
    )

In [14]:
# ============================================================
# CELL 15 — MULTICLASS METRIC-SPACE VALIDATION
# ============================================================

def evaluate_multiclass(
    model,
    reference_loader,
    val_loader
):

    # ========================================================
    # EMBEDDINGS
    # ========================================================

    (
        reference_embeddings,
        reference_labels
    ) = extract_h_embeddings(

        model,
        reference_loader
    )


    (
        val_embeddings,
        val_labels
    ) = extract_h_embeddings(

        model,
        val_loader
    )


    classes = np.array(

        sorted(
            np.unique(
                reference_labels
            )
        ),

        dtype=np.int64
    )


    # ========================================================
    # PROTOTYPES
    # ========================================================

    prototypes = []


    for class_id in classes:

        class_embeddings = (

            reference_embeddings[
                reference_labels
                ==
                class_id
            ]
        )


        prototype = (
            class_embeddings
            .mean(
                axis=0
            )
        )


        prototype = (

            prototype

            /

            (
                np.linalg.norm(
                    prototype
                )
                +
                1e-12
            )
        )


        prototypes.append(
            prototype
        )


    prototypes = np.stack(
        prototypes,
        axis=0
    ).astype(
        np.float32
    )


    # ========================================================
    # COSINE SIMILARITY
    # ========================================================

    similarities = (

        val_embeddings

        @

        prototypes.T
    )


    prediction_indices = (

        similarities
        .argmax(
            axis=1
        )
    )


    y_pred = (

        classes[
            prediction_indices
        ]
    )


    # ========================================================
    # METRICS
    # ========================================================

    metrics = {

        "accuracy":
            float(
                accuracy_score(
                    val_labels,
                    y_pred
                )
            ),

        "macro_f1":
            float(
                f1_score(
                    val_labels,
                    y_pred,
                    average="macro",
                    zero_division=0
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    val_labels,
                    y_pred
                )
            )
    }


    return (
        metrics,
        val_labels,
        y_pred
    )

In [15]:
# ============================================================
# CELL 16 — RANDOM INITIALIZATION BASELINE
# ============================================================

(
    initial_val_metrics,
    _,
    _
) = evaluate_multiclass(

    model=
        model,

    reference_loader=
        reference_loader,

    val_loader=
        val_loader
)


print("=" * 70)
print("SUPCON V3 — BEFORE TRAINING")
print("=" * 70)

print(
    "Accuracy:",
    f"{initial_val_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{initial_val_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{initial_val_metrics['balanced_accuracy']:.4f}"
)

SUPCON V3 — BEFORE TRAINING
Accuracy: 0.2218
Macro-F1: 0.1906
Balanced Accuracy: 0.2155


In [16]:
# ============================================================
# CELL 17 — OPTIMIZER + AMP
# ============================================================

try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY,

        fused=
            DEVICE.type == "cuda"
    )


    fused_adamw = (
        DEVICE.type == "cuda"
    )


except (
    TypeError,
    RuntimeError
):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY
    )


    fused_adamw = False


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=
        AMP_ENABLED
)


print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    AMP_ENABLED
)

print(
    "Fused AdamW:",
    fused_adamw
)

Learning rate: 0.0003
Weight decay: 0.0001
AMP: True
Fused AdamW: True


In [17]:
# ============================================================
# CELL 18 — TRAIN ONE SUPCON EPOCH
# ============================================================

def train_supcon_epoch(
    model,
    loader,
    sampler,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()


    sampler.set_epoch(
        epoch
    )


    total_loss = 0.0

    total_positive_cosine = 0.0

    total_negative_cosine = 0.0

    total_cosine_gap = 0.0

    total_h_std = 0.0

    total_batches = 0


    start_time = (
        time.perf_counter()
    )


    for batch in loader:

        x = (
            batch["x"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        labels = (
            batch["class_id"]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # FORWARD
        # ====================================================

        with torch.autocast(

            device_type=
                DEVICE.type,

            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,

            enabled=
                AMP_ENABLED

        ):

            h, z = model(
                x
            )


        # SupCon loss viene calcolata in FP32
        (
            loss,
            stats
        ) = criterion(
            z,
            labels
        )


        # ====================================================
        # BACKWARD
        # ====================================================

        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            loss.backward()

            optimizer.step()


        # ====================================================
        # DIAGNOSTICS
        # ====================================================

        h_std = (

            h.float()
            .std(
                dim=0
            )
            .mean()
        )


        total_loss += (
            loss.detach().item()
        )


        total_positive_cosine += (
            stats[
                "positive_cosine"
            ].item()
        )


        total_negative_cosine += (
            stats[
                "negative_cosine"
            ].item()
        )


        total_cosine_gap += (
            stats[
                "cosine_gap"
            ].item()
        )


        total_h_std += (
            h_std.detach().item()
        )


        total_batches += 1


    elapsed = (

        time.perf_counter()
        -
        start_time
    )


    return {

        "loss":
            total_loss
            /
            total_batches,

        "positive_cosine":
            total_positive_cosine
            /
            total_batches,

        "negative_cosine":
            total_negative_cosine
            /
            total_batches,

        "cosine_gap":
            total_cosine_gap
            /
            total_batches,

        "h_std":
            total_h_std
            /
            total_batches,

        "seconds":
            elapsed
    }

In [18]:
# ============================================================
# CELL 19 — SUPCON SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


# Backup iniziale
initial_model_state = copy.deepcopy(
    model.state_dict()
)


print("=" * 92)
print("SUPCON V3 — TUMOR + HEALTHY — SMOKE TEST")
print("=" * 92)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = train_supcon_epoch(

        model=
            model,

        loader=
            train_loader,

        sampler=
            train_sampler,

        criterion=
            supcon_criterion,

        optimizer=
            optimizer,

        scaler=
            scaler,

        epoch=
            epoch
    )


    # ========================================================
    # VALIDATION MULTICLASS
    # ========================================================

    (
        val_metrics,
        _,
        _
    ) = evaluate_multiclass(

        model=
            model,

        reference_loader=
            reference_loader,

        val_loader=
            val_loader
    )


    print(

        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | loss "
        f"{train_metrics['loss']:.4f}"

        f" | cos+ "
        f"{train_metrics['positive_cosine']:.4f}"

        f" | cos- "
        f"{train_metrics['negative_cosine']:.4f}"

        f" | gap "
        f"{train_metrics['cosine_gap']:.4f}"

        f" | h_std "
        f"{train_metrics['h_std']:.5f}"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

SUPCON V3 — TUMOR + HEALTHY — SMOKE TEST
Epoch 01/3 | loss 4.4691 | cos+ 0.9742 | cos- 0.9598 | gap 0.0144 | h_std 0.01554 | val Acc 0.2484 | val F1 0.2145 | bal Acc 0.2629 | 22.6s
Epoch 02/3 | loss 4.4378 | cos+ 0.9649 | cos- 0.9456 | gap 0.0193 | h_std 0.02288 | val Acc 0.2691 | val F1 0.2317 | bal Acc 0.2813 | 7.0s
Epoch 03/3 | loss 4.4213 | cos+ 0.9606 | cos- 0.9390 | gap 0.0217 | h_std 0.02606 | val Acc 0.2780 | val F1 0.2369 | bal Acc 0.2871 | 6.1s


In [19]:
# ============================================================
# CELL 20 — RESTORE BEFORE FULL SUPCON TRAINING
# ============================================================

model.load_state_dict(
    initial_model_state,
    strict=True
)


# Nuovo optimizer:
# nessuno stato ereditato dallo smoke test
try:

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY,

        fused=(DEVICE.type == "cuda")
    )

except (TypeError, RuntimeError):

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=LEARNING_RATE,

        weight_decay=WEIGHT_DECAY
    )


scaler = torch.amp.GradScaler(

    "cuda",

    enabled=AMP_ENABLED
)


print(
    "Modello ripristinato allo stato iniziale."
)

print(
    "Optimizer reinizializzato."
)

print(
    "LR:",
    LEARNING_RATE
)

Modello ripristinato allo stato iniziale.
Optimizer reinizializzato.
LR: 0.0003


In [20]:
# ============================================================
# CELL 21 — FULL TRAINING CONFIG
# ============================================================

MAX_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 8

MIN_DELTA = 1e-4


BEST_CHECKPOINT_PATH = (
    ARTIFACTS_DIR
    / "supcon_v3_tumor_healthy_best.pt"
)


HISTORY_PATH = (
    ARTIFACTS_DIR
    / "supcon_v3_tumor_healthy_history.tsv"
)


print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Patience:",
    EARLY_STOPPING_PATIENCE
)

print(
    "Checkpoint metric: validation Macro-F1"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

Max epochs: 40
Patience: 8
Checkpoint metric: validation Macro-F1
Checkpoint: D:\Daria\Desktop\eccdna_fcgr_siamese\artifacts\supcon_v3_tumor_healthy\supcon_v3_tumor_healthy_best.pt


In [21]:
# ============================================================
# CELL 22 — FULL SUPCON TRAINING
# ============================================================

best_val_macro_f1 = -np.inf

best_epoch = 0

epochs_without_improvement = 0

history = []


print("=" * 100)
print("SUPCON V3 — TUMOR + HEALTHY — FULL TRAINING")
print("=" * 100)


for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    # ========================================================
    # TRAIN
    # ========================================================

    train_metrics = train_supcon_epoch(

        model=model,

        loader=train_loader,

        sampler=train_sampler,

        criterion=supcon_criterion,

        optimizer=optimizer,

        scaler=scaler,

        epoch=epoch
    )


    # ========================================================
    # MULTICLASS VALIDATION
    # ========================================================

    (
        val_metrics,
        _,
        _
    ) = evaluate_multiclass(

        model=model,

        reference_loader=reference_loader,

        val_loader=val_loader
    )


    current_macro_f1 = float(
        val_metrics[
            "macro_f1"
        ]
    )


    improved = (

        current_macro_f1

        >

        best_val_macro_f1
        +
        MIN_DELTA
    )


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if improved:

        best_val_macro_f1 = (
            current_macro_f1
        )

        best_epoch = (
            epoch
        )

        epochs_without_improvement = 0


        checkpoint = {

            "epoch":
                epoch,

            "best_epoch":
                epoch,

            "best_val_macro_f1":
                current_macro_f1,

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "embedding_dim":
                EMBEDDING_DIM,

            "projection_dim":
                PROJECTION_DIM,

            "temperature":
                TEMPERATURE,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "samples_per_class":
                SAMPLES_PER_CLASS,

            "n_classes":
                N_CLASSES,

            "k":
                K,

            "task":
                "tumor_healthy",

            "loss":
                "supervised_contrastive",

            "positive_cosine":
                train_metrics[
                    "positive_cosine"
                ],

            "negative_cosine":
                train_metrics[
                    "negative_cosine"
                ],

            "cosine_gap":
                train_metrics[
                    "cosine_gap"
                ],

            "h_std":
                train_metrics[
                    "h_std"
                ]
        }


        torch.save(
            checkpoint,
            BEST_CHECKPOINT_PATH
        )


    else:

        epochs_without_improvement += 1


    # ========================================================
    # HISTORY
    # ========================================================

    history.append(
        {

            "epoch":
                epoch,

            "train_loss":
                train_metrics[
                    "loss"
                ],

            "positive_cosine":
                train_metrics[
                    "positive_cosine"
                ],

            "negative_cosine":
                train_metrics[
                    "negative_cosine"
                ],

            "cosine_gap":
                train_metrics[
                    "cosine_gap"
                ],

            "h_std":
                train_metrics[
                    "h_std"
                ],

            "val_accuracy":
                val_metrics[
                    "accuracy"
                ],

            "val_macro_f1":
                val_metrics[
                    "macro_f1"
                ],

            "val_balanced_accuracy":
                val_metrics[
                    "balanced_accuracy"
                ],

            "seconds":
                train_metrics[
                    "seconds"
                ]
        }
    )


    marker = (
        " *BEST*"
        if improved
        else ""
    )


    print(

        f"Epoch {epoch:02d}/{MAX_EPOCHS}"

        f" | loss "
        f"{train_metrics['loss']:.4f}"

        f" | cos+ "
        f"{train_metrics['positive_cosine']:.4f}"

        f" | cos- "
        f"{train_metrics['negative_cosine']:.4f}"

        f" | gap "
        f"{train_metrics['cosine_gap']:.4f}"

        f" | h_std "
        f"{train_metrics['h_std']:.5f}"

        f" | val Acc "
        f"{val_metrics['accuracy']:.4f}"

        f" | val F1 "
        f"{val_metrics['macro_f1']:.4f}"

        f" | bal Acc "
        f"{val_metrics['balanced_accuracy']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"

        f"{marker}"
    )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print()
        print(
            f"Early stopping at epoch {epoch}."
        )

        break


# ============================================================
# SAVE HISTORY
# ============================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(

    HISTORY_PATH,

    sep="\t",

    index=False
)


print()
print("=" * 100)
print("SUPCON TRAINING COMPLETATO")
print("=" * 100)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best Val Macro-F1:",
    f"{best_val_macro_f1:.6f}"
)

print(
    "Checkpoint:",
    BEST_CHECKPOINT_PATH
)

print(
    "History:",
    HISTORY_PATH
)

SUPCON V3 — TUMOR + HEALTHY — FULL TRAINING
Epoch 01/40 | loss 4.4691 | cos+ 0.9742 | cos- 0.9598 | gap 0.0144 | h_std 0.01554 | val Acc 0.2484 | val F1 0.2145 | bal Acc 0.2629 | 6.1s *BEST*
Epoch 02/40 | loss 4.4378 | cos+ 0.9649 | cos- 0.9456 | gap 0.0193 | h_std 0.02288 | val Acc 0.2691 | val F1 0.2317 | bal Acc 0.2813 | 6.2s *BEST*
Epoch 03/40 | loss 4.4213 | cos+ 0.9606 | cos- 0.9390 | gap 0.0217 | h_std 0.02606 | val Acc 0.2780 | val F1 0.2369 | bal Acc 0.2871 | 6.2s *BEST*
Epoch 04/40 | loss 4.4039 | cos+ 0.9557 | cos- 0.9316 | gap 0.0241 | h_std 0.02961 | val Acc 0.2765 | val F1 0.2352 | bal Acc 0.2859 | 6.2s
Epoch 05/40 | loss 4.4022 | cos+ 0.9539 | cos- 0.9295 | gap 0.0244 | h_std 0.03147 | val Acc 0.2759 | val F1 0.2365 | bal Acc 0.2871 | 5.8s
Epoch 06/40 | loss 4.3914 | cos+ 0.9523 | cos- 0.9263 | gap 0.0260 | h_std 0.03295 | val Acc 0.2826 | val F1 0.2428 | bal Acc 0.2959 | 18.7s *BEST*
Epoch 07/40 | loss 4.3875 | cos+ 0.9518 | cos- 0.9252 | gap 0.0266 | h_std 0.03409 | va

In [22]:
# ============================================================
# CELL 23 — RELOAD BEST SUPCON CHECKPOINT
# ============================================================

best_checkpoint = torch.load(

    BEST_CHECKPOINT_PATH,

    map_location=DEVICE
)


model.load_state_dict(

    best_checkpoint[
        "model_state_dict"
    ],

    strict=True
)


model.eval()


print("=" * 72)
print("BEST SUPCON CHECKPOINT")
print("=" * 72)

print(
    "Epoch:",
    best_checkpoint[
        "best_epoch"
    ]
)

print(
    "Val Macro-F1:",
    f"{best_checkpoint['best_val_macro_f1']:.6f}"
)

print(
    "Val Accuracy:",
    f"{best_checkpoint['val_accuracy']:.6f}"
)

print(
    "Val Balanced Accuracy:",
    f"{best_checkpoint['val_balanced_accuracy']:.6f}"
)

print(
    "Train cosine gap:",
    f"{best_checkpoint['cosine_gap']:.6f}"
)

print(
    "h_std:",
    f"{best_checkpoint['h_std']:.6f}"
)

BEST SUPCON CHECKPOINT
Epoch: 26
Val Macro-F1: 0.270669
Val Accuracy: 0.303701
Val Balanced Accuracy: 0.302147
Train cosine gap: 0.073510
h_std: 0.059006


In [23]:
# ============================================================
# CELL 24 — FULL TRAIN REFERENCE SET
# ============================================================

full_reference_dataset = SingleFCGRDataset(

    metadata=
        train_metadata,

    fcgr_memmap=
        fcgr_memmap,

    id_to_row=
        id_to_fcgr_row
)


full_reference_loader = DataLoader(

    full_reference_dataset,

    batch_size=
        EVAL_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=
        torch.cuda.is_available()
)


print(
    "Full train reference:",
    len(full_reference_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

Full train reference: 96167
Validation: 9753


In [24]:
# ============================================================
# CELL 25 — FINAL FULL-REFERENCE PROTOTYPE EVALUATION
# ============================================================

(
    final_supcon_metrics,
    final_y_true,
    final_y_pred
) = evaluate_multiclass(

    model=model,

    reference_loader=
        full_reference_loader,

    val_loader=
        val_loader
)


print("=" * 78)
print("SUPCON V3 — FULL TRAIN PROTOTYPES")
print("=" * 78)

print(
    "Accuracy:",
    f"{final_supcon_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{final_supcon_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{final_supcon_metrics['balanced_accuracy']:.4f}"
)

SUPCON V3 — FULL TRAIN PROTOTYPES
Accuracy: 0.3017
Macro-F1: 0.2703
Balanced Accuracy: 0.3020


In [25]:
# ============================================================
# CELL 26 — FINAL COMPARISON
# ============================================================

comparison_df = pd.DataFrame(
    [
        {
            "experiment":
                "Euclidean 18-way → evaluated 12-way",

            "accuracy":
                0.2808,

            "macro_f1":
                0.2481,

            "balanced_accuracy":
                0.3025
        },

        {
            "experiment":
                "Frozen Euclidean embedding + linear probe",

            "accuracy":
                0.3071,

            "macro_f1":
                0.2796,

            "balanced_accuracy":
                0.3223
        },

        {
            "experiment":
                "Euclidean trained directly 12-way",

            "accuracy":
                0.279606,

            "macro_f1":
                0.243436,

            "balanced_accuracy":
                0.295295
        },

        {
            "experiment":
                "SupCon V3 trained 12-way",

            "accuracy":
                final_supcon_metrics[
                    "accuracy"
                ],

            "macro_f1":
                final_supcon_metrics[
                    "macro_f1"
                ],

            "balanced_accuracy":
                final_supcon_metrics[
                    "balanced_accuracy"
                ]
        }
    ]
)


comparison_df = (
    comparison_df
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    comparison_df
)

,experiment,accuracy,macro_f1,balanced_accuracy
0,Frozen Euclidean embedding + linear probe,0.307100,0.279600,0.322300
1,SupCon V3 trained 12-way,0.301651,0.270285,0.301969
2,Euclidean 18-way → evaluated 12-way,0.280800,0.248100,0.302500
3,Euclidean trained directly 12-way,0.279606,0.243436,0.295295


In [26]:
# ============================================================
# CELL 27 — EXTRACT SUPCON PROJECTION EMBEDDINGS z
# ============================================================

def extract_z_embeddings(
    model,
    loader
):

    model.eval()

    embeddings_list = []
    labels_list = []

    with torch.no_grad():

        for batch in loader:

            x = (
                batch["x"]
                .to(
                    DEVICE,
                    non_blocking=True
                )
            )

            with torch.autocast(

                device_type=DEVICE.type,

                dtype=(
                    torch.float16
                    if DEVICE.type == "cuda"
                    else torch.bfloat16
                ),

                enabled=AMP_ENABLED

            ):

                _, z = model(x)

            embeddings_list.append(
                z.float()
                .cpu()
                .numpy()
            )

            labels_list.append(
                batch["class_id"]
                .numpy()
            )

    return (
        np.concatenate(
            embeddings_list,
            axis=0
        ),

        np.concatenate(
            labels_list,
            axis=0
        ).astype(np.int64)
    )


print(
    "Estrazione train z..."
)

train_z, train_z_labels = (
    extract_z_embeddings(
        model,
        full_reference_loader
    )
)


print(
    "Estrazione validation z..."
)

val_z, val_z_labels = (
    extract_z_embeddings(
        model,
        val_loader
    )
)


print(
    "Train z:",
    train_z.shape
)

print(
    "Val z:",
    val_z.shape
)

print(
    "Norma media:",
    np.linalg.norm(
        train_z,
        axis=1
    ).mean()
)

Estrazione train z...
Estrazione validation z...
Train z: (96167, 128)
Val z: (9753, 128)
Norma media: 1.0


In [27]:
# ============================================================
# CELL 28 — PROTOTYPE CLASSIFICATION IN z SPACE
# ============================================================

classes_z = np.array(
    sorted(
        np.unique(
            train_z_labels
        )
    ),
    dtype=np.int64
)


z_prototypes = []


for class_id in classes_z:

    class_embeddings = (
        train_z[
            train_z_labels
            ==
            class_id
        ]
    )


    prototype = (
        class_embeddings
        .mean(axis=0)
    )


    prototype = (
        prototype
        /
        (
            np.linalg.norm(
                prototype
            )
            +
            1e-12
        )
    )


    z_prototypes.append(
        prototype
    )


z_prototypes = np.stack(
    z_prototypes,
    axis=0
).astype(np.float32)


# cosine similarity
z_similarities = (
    val_z
    @
    z_prototypes.T
)


z_prediction_indices = (
    z_similarities.argmax(
        axis=1
    )
)


z_y_pred = (
    classes_z[
        z_prediction_indices
    ]
)


z_metrics = {

    "accuracy":
        accuracy_score(
            val_z_labels,
            z_y_pred
        ),

    "macro_f1":
        f1_score(
            val_z_labels,
            z_y_pred,
            average="macro",
            zero_division=0
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            val_z_labels,
            z_y_pred
        )
}


print("=" * 76)
print("SUPCON — PROJECTION SPACE z")
print("=" * 76)

print(
    "Accuracy:",
    f"{z_metrics['accuracy']:.4f}"
)

print(
    "Macro-F1:",
    f"{z_metrics['macro_f1']:.4f}"
)

print(
    "Balanced Accuracy:",
    f"{z_metrics['balanced_accuracy']:.4f}"
)

SUPCON — PROJECTION SPACE z
Accuracy: 0.2979
Macro-F1: 0.2679
Balanced Accuracy: 0.2994


In [28]:
comparison_h_z = pd.DataFrame(
    [
        {
            "space": "SupCon encoder h",
            "accuracy":
                final_supcon_metrics["accuracy"],
            "macro_f1":
                final_supcon_metrics["macro_f1"],
            "balanced_accuracy":
                final_supcon_metrics["balanced_accuracy"]
        },

        {
            "space": "SupCon projection z",
            "accuracy":
                z_metrics["accuracy"],
            "macro_f1":
                z_metrics["macro_f1"],
            "balanced_accuracy":
                z_metrics["balanced_accuracy"]
        }
    ]
)


display(
    comparison_h_z
)

,space,accuracy,macro_f1,balanced_accuracy
0,SupCon encoder h,0.301651,0.270285,0.301969
1,SupCon projection z,0.297857,0.267857,0.299420
